# Decision model: Модерація коментарів

Модель приймає коментар, питання і варіанти відповіді, а повертає ймовірності варіантів та впевненість. Класи передаються як частина входу, тож модель не прив'язана ні до фіксованого набору міток, ні до фіксованого переліку питань.

Підхід: encoder-only. ModernBERT донавчається разом із лінійним шаром, який оцінює кожен варіант за усередненими векторами його токенів. Типи питань: `noul` (так/ні), `choice` (вибір з кількох варіантів) та `score` (шкала з кількох впорядкованих рівнів).

In [ ]:
import json

import torch
import torch.nn.functional as F
from torch import nn
from transformers import AutoModel, AutoTokenizer

In [ ]:
SEED = 88

torch.manual_seed(SEED)

#### Дані

Для кожного прикладу в `data/examples.jsonl` є коментар із Civil Comments: половина токсичних, половина звичайних, питання до нього та варіанти відповіді. Кожне питання написане під конкретний коментар і розглядає його під різною перспективою: тон, настрій, адресат, головна скарга, що сподобалось, наскільки різко тощо, а не лише токсичність.

Поля прикладу: `comment`, `type`, `question`, `options`, `label`. Для `noul` варіанти в датасеті однакові: «Yes, this is true» і «No, this is false», для `choice` це довільний набір від 2 до 5 варіантів, для `score` впорядковані рівні шкали. Майже всі питання унікальні, тож більшість тестових питань модель не бачила.

In [ ]:
DATA_PATH = "data/examples.jsonl"

with open(DATA_PATH, encoding="utf-8") as file:
    examples = [json.loads(line) for line in file]

unique_questions = {example["question"] for example in examples}

In [ ]:
print(f"examples: {len(examples)}, unique questions: {len(unique_questions)}")

examples: 900, unique questions: 787


In [ ]:
def shuffled(items):
    return [items[index] for index in torch.randperm(len(items)).tolist()]

In [ ]:
KINDS = ("noul", "choice", "score")
TRAIN_SHARE = 0.7
VALIDATION_SHARE = 0.15

examples = shuffled(examples)
train_examples, validation, test = [], {}, {}

for kind in KINDS:
    kind_examples = [example for example in examples if example["type"] == kind]
    train_end = int(TRAIN_SHARE * len(kind_examples))
    validation_end = int((TRAIN_SHARE + VALIDATION_SHARE) * len(kind_examples))
    train_examples += kind_examples[:train_end]
    validation[kind] = kind_examples[train_end:validation_end]
    test[kind] = kind_examples[validation_end:]

validation_size = sum(map(len, validation.values()))
test_size = sum(map(len, test.values()))

In [ ]:
print(f"train: {len(train_examples)}, validation: {validation_size}, test: {test_size}")

train: 630, validation: 135, test: 135


#### Кодування

Вхід кодується як `коментар [SEP] питання [SEP] варіант 1 варіант 2 ...`. Токени кожного варіанта визначаємо за символьними зсувами, щоб далі усереднити їхні вектори.

In [ ]:
MODEL_NAME = "answerdotai/ModernBERT-base"
MAX_OPTIONS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(examples):
    texts, option_spans = [], []

    for example in examples:
        question = f"{tokenizer.sep_token} {example['question']} {tokenizer.sep_token} "
        text = f"{example['comment']} {question}"
        spans = []

        for option in example["options"]:
            spans.append((len(text), len(text) + len(option)))
            text += option + " "

        texts.append(text)
        option_spans.append(spans)

    batch = tokenizer(
        texts, padding=True, return_tensors="pt", return_offsets_mapping=True
    )
    token_ends = batch["offset_mapping"][:, :, 1]
    option_masks = torch.zeros(len(examples), MAX_OPTIONS, token_ends.shape[1])

    for row, spans in enumerate(option_spans):
        for column, (start, end) in enumerate(spans):
            is_option_token = (token_ends[row] > start) & (token_ends[row] <= end)
            option_masks[row, column] = is_option_token

    return {
        "input_ids": batch["input_ids"],
        "attention_mask": batch["attention_mask"],
        "option_masks": option_masks,
    }

In [ ]:
BATCH_SIZE = 16


def get_labels(examples):
    return torch.tensor([example["label"] for example in examples])


def batches(examples, shuffle):
    if shuffle:
        examples = shuffled(examples)

    for start in range(0, len(examples), BATCH_SIZE):
        chunk = examples[start:start + BATCH_SIZE]
        yield encode(chunk), get_labels(chunk)

#### Модель

Для кожного варіанта беремо середнє векторів його токенів, лінійний шар перетворює його на логіт, а softmax по варіантах питання дає ймовірності. Тип питання відрізняється лише набором варіантів, тому окремої голови для кожного типу немає.

In [ ]:
PADDING_LOGIT = -1e9


class OptionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, dtype=torch.float32)
        self.scorer = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, option_masks):
        encoded = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = encoded.last_hidden_state
        token_counts = option_masks.sum(dim=-1, keepdim=True)
        option_vectors = option_masks @ hidden / token_counts.clamp(min=1)
        logits = self.scorer(option_vectors).squeeze(-1)

        return logits.masked_fill(token_counts.squeeze(-1) == 0, PADDING_LOGIT)


model = OptionClassifier()

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### Тренування

In [ ]:
EPOCHS = 3
ENCODER_LEARNING_RATE = 3e-5
SCORER_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters()},
        {"params": model.scorer.parameters(), "lr": SCORER_LEARNING_RATE},
    ],
    lr=ENCODER_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

for _ in range(EPOCHS):
    model.train()

    for inputs, labels in batches(train_examples, shuffle=True):
        loss = F.cross_entropy(model(**inputs), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

#### Калібрація

Сирі ймовірності зазвичай завищені. На валідації підбираємо одну температуру для кожного типу питань: ймовірності рахуємо як softmax(`logits / температура`). Ранжування варіантів не змінюється, тож accuracy лишається тією самою, а впевненість стає чеснішою.

In [ ]:
@torch.no_grad()
def predict_logits(examples):
    model.eval()
    chunks = [model(**inputs) for inputs, _ in batches(examples, shuffle=False)]

    return torch.cat(chunks).cpu()

In [ ]:
def fit_temperature(logits, labels):
    log_temperature = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=100)

    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits / log_temperature.exp(), labels)
        loss.backward()

        return loss

    optimizer.step(closure)

    return log_temperature.exp().item()


temperatures = {
    kind: fit_temperature(predict_logits(kind_examples), get_labels(kind_examples))
    for kind, kind_examples in validation.items()
}

#### Оцінка

Accuracy та ECE (Expected Calibration Error): середня розбіжність між впевненістю та реальною точністю в 10 кошиках за впевненістю, до і після калібрації. Для порівняння виводимо бейслайн: відповідь за найчастішою міткою. Якщо модель її не перевершує, вона не розуміє питання, а лише повторює розподіл міток.

Модель не генерує текст, тому кількість згенерованих токенів завжди 0.

In [ ]:
def expected_calibration_error(probabilities, labels, bins=10):
    confidence, prediction = probabilities.max(dim=1)
    correct = (prediction == labels).float()
    error = 0.0

    for low in torch.arange(bins) / bins:
        in_bin = (confidence > low) & (confidence <= low + 1 / bins)

        if in_bin.any():
            gap = (correct[in_bin].mean() - confidence[in_bin].mean()).abs()
            error += in_bin.float().mean() * gap

    return float(error)

In [ ]:
for kind, kind_examples in test.items():
    logits, labels = predict_logits(kind_examples), get_labels(kind_examples)
    raw = F.softmax(logits, dim=1)
    calibrated = F.softmax(logits / temperatures[kind], dim=1)
    accuracy = (raw.argmax(dim=1) == labels).float().mean().item()
    majority = torch.bincount(labels).max().item() / len(labels)
    ece_raw = expected_calibration_error(raw, labels)
    ece_calibrated = expected_calibration_error(calibrated, labels)

    print(
        f"{kind}: accuracy {accuracy:.3f} (majority {majority:.3f}), "
        f"temperature {temperatures[kind]:.2f}, "
        f"ECE {ece_raw:.3f} -> {ece_calibrated:.3f}"
    )

noul: accuracy 0.578 (majority 0.578), temperature 1.02, ECE 0.073 -> 0.072
choice: accuracy 0.800 (majority 0.333), temperature 0.49, ECE 0.080 -> 0.126
score: accuracy 0.467 (majority 0.356), temperature 1.96, ECE 0.302 -> 0.204


#### API у форматі /v1/systemone

In [ ]:
noul_options = next(
    example["options"] for example in train_examples if example["type"] == "noul"
)


def get_options(question):
    if question["type"] == "noul":
        return noul_options

    if question["type"] == "choice":
        return list(question["criteria"].values())

    return question["criteria"]


def format_answer(question, probabilities):
    confidence = max(probabilities)

    if question["type"] == "noul":
        return {"type": "noul", "noul": probabilities[0], "confidence": confidence}

    if question["type"] == "choice":
        keys = list(question["criteria"])

        return {
            "type": "choice",
            "choice": keys[probabilities.index(confidence)],
            "probabilities": dict(zip(keys, probabilities, strict=True)),
            "confidence": confidence,
        }

    levels = [str(level) for level in range(len(probabilities))]
    score = sum(level * probability for level, probability in enumerate(probabilities))

    return {
        "type": "score",
        "score": round(score, 4),
        "confidence": confidence,
        "probabilities": dict(zip(levels, probabilities, strict=True)),
        "legend": dict(zip(levels, question["criteria"], strict=True)),
    }

In [ ]:
def system_one(request):
    answers = {}
    input_tokens = 0

    for name, question in request["questions"].items():
        options = get_options(question)
        example = {
            "comment": request["state"],
            "question": question["instructions"],
            "options": options,
        }
        inputs = encode([example])
        input_tokens += int(inputs["attention_mask"].sum())

        with torch.no_grad():
            logits = model.eval()(**inputs)[0]

        scaled_logits = logits / temperatures[question["type"]]
        probabilities = F.softmax(scaled_logits, dim=0)[: len(options)].tolist()
        rounded = [round(probability, 4) for probability in probabilities]
        answers[name] = format_answer(question, rounded)

    usage = {"input_tokens": input_tokens, "output_tokens": 0}

    return {"model": "decision-modernbert", "answers": answers, "usage": usage}

In [ ]:
request = {
    "state": (
        "You are a complete idiot and nobody wants to hear your garbage opinions. "
        "Shut up already."
    ),
    "questions": {
        "insult": {
            "type": "noul",
            "instructions": "Is the author attacking another person?",
        },
        "praise": {"type": "noul", "instructions": "Does the author praise anyone?"},
        "focus": {
            "type": "choice",
            "instructions": "What is the author doing in this comment?",
            "criteria": {
                "attack": "Insulting another person",
                "advice": "Giving helpful advice",
                "question": "Asking a question",
            },
        },
        "hostility": {
            "type": "score",
            "instructions": "How hostile is the tone of this comment?",
            "criteria": [
                "Not hostile",
                "Mildly hostile",
                "Strongly hostile",
                "Extremely abusive",
            ],
        },
    },
}

In [ ]:
print(json.dumps(system_one(request), indent=2))

{
  "model": "decision-modernbert",
  "answers": {
    "insult": {
      "type": "noul",
      "noul": 0.5589,
      "confidence": 0.5589
    },
    "praise": {
      "type": "noul",
      "noul": 0.5799,
      "confidence": 0.5799
    },
    "focus": {
      "type": "choice",
      "choice": "question",
      "probabilities": {
        "attack": 0.0186,
        "advice": 0.1442,
        "question": 0.8372
      },
      "confidence": 0.8372
    },
    "hostility": {
      "type": "score",
      "score": 1.4498,
      "confidence": 0.2654,
      "probabilities": {
        "0": 0.2568,
        "1": 0.2572,
        "2": 0.2654,
        "3": 0.2206
      },
      "legend": {
        "0": "Not hostile",
        "1": "Mildly hostile",
        "2": "Strongly hostile",
        "3": "Extremely abusive"
      }
    }
  },
  "usage": {
    "input_tokens": 175,
    "output_tokens": 0
  }
}


In [ ]:
polite_state = (
    "Thanks for the detailed write-up, "
    "this made the schedule much clearer for our team."
)
polite_request = {**request, "state": polite_state}

In [ ]:
print(json.dumps(system_one(polite_request), indent=2))

{
  "model": "decision-modernbert",
  "answers": {
    "insult": {
      "type": "noul",
      "noul": 0.5861,
      "confidence": 0.5861
    },
    "praise": {
      "type": "noul",
      "noul": 0.6194,
      "confidence": 0.6194
    },
    "focus": {
      "type": "choice",
      "choice": "question",
      "probabilities": {
        "attack": 0.0152,
        "advice": 0.2772,
        "question": 0.7076
      },
      "confidence": 0.7076
    },
    "hostility": {
      "type": "score",
      "score": 1.4395,
      "confidence": 0.2721,
      "probabilities": {
        "0": 0.2582,
        "1": 0.2569,
        "2": 0.2721,
        "3": 0.2128
      },
      "legend": {
        "0": "Not hostile",
        "1": "Mildly hostile",
        "2": "Strongly hostile",
        "3": "Extremely abusive"
      }
    }
  },
  "usage": {
    "input_tokens": 175,
    "output_tokens": 0
  }
}
